# Candidate generator A/B on Colab
Compares the **baseline** generator (repo + `prune_candidate_generator`) against the **capped exact_name** variant
(friend's `origin/candidate/updated`) through the full pipeline (same 10k dev S1 sample, seed 42).

**Before running, put these in Google Drive under `MyDrive/MLChallenge/`:**
- `colab_code.zip` (project code snapshot)
- `dataset.zip` **or** the unzipped `dataset/` folder (must contain `train/` and `test/`)

Use a **High-RAM** runtime if you can (Runtime > Change runtime type). Free tier may be tight for the 636k pool.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
rm -rf /content/work && mkdir -p /content/work/student_resource && cd /content/work
unzip -q /content/drive/MyDrive/MLChallenge/colab_code.zip
D=/content/drive/MyDrive/MLChallenge
if [ -f $D/dataset.zip ]; then
  unzip -q $D/dataset.zip -d student_resource
elif [ -d $D/dataset ]; then
  cp -r $D/dataset student_resource/dataset     # copy to local disk: much faster than reading from Drive
fi
# tolerate one extra nesting level (dataset/dataset/train)
if [ ! -d student_resource/dataset/train ] && [ -d student_resource/dataset/dataset/train ]; then
  mv student_resource/dataset/dataset/* student_resource/dataset/ && rmdir student_resource/dataset/dataset
fi
find student_resource -maxdepth 3 | head -20
pip -q install rapidfuzz xgboost pyyaml psutil
nproc; free -h | head -2


Check the tree above: the config expects `student_resource/dataset/train/train_source1.tsv` etc.
If the zip unpacks one level deeper or shallower, move it so that path exists before continuing.

In [ ]:
import os, subprocess, shutil, json, collections
os.chdir('/content/work')
assert os.path.exists('student_resource/dataset/train/train_source1.tsv'), 'dataset path mismatch, see note above'
NW = max(1, os.cpu_count() or 1)   # Colab usually has 2 cores

def run(label, notes):
    """Run the pipeline and stream its output into this cell (Colab drops raw subprocess output)."""
    os.makedirs('results', exist_ok=True)
    log = f'results/{label}_run.log'
    cmd = ['python', '-u', '-m', 'src.pipeline', '--config', 'configs/baseline.yaml',
           '--workers', str(NW), '--notes', notes]
    print('>>>', ' '.join(cmd), flush=True)
    tail = collections.deque(maxlen=40)
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line); tail.append(line)
        rc = p.wait()
    if rc != 0:
        print(f'\n=== {label} FAILED, returncode={rc} (negative / 137 = killed, usually out of memory) ===')
        print(subprocess.run('free -h | head -2', shell=True, capture_output=True, text=True).stdout)
        raise RuntimeError(f'{label} failed')
    shutil.copy('outputs/baseline_metrics.json', f'results/{label}_metrics.json')
    shutil.copy('experiments/experiment_log.csv', f'results/{label}_experiment_log.csv')


## Shrink the candidate pool to fit Colab memory
The full pool is 10.3M records (5.0M S2 + 5.3M S3); indexing it needs far more than the 12 GB of a standard Colab runtime
(Run A was killed with returncode -9). `dev.pool_sample_frac` keeps **all true matches** of the sampled S1 entities plus a random
fraction of the other records, and the same sample is used for A and B (seed 42), so the A/B comparison is fair.
Absolute scores will be optimistic (fewer look-alike records); use only the **difference** between A and B.
Set `POOL_FRAC = 1.0` only on a High-RAM runtime.

In [ ]:
import re
POOL_FRAC = 0.10
txt = open('configs/baseline.yaml').read()
txt, n = re.subn(r'(pool_sample_frac:\s*)[0-9.]+', lambda m: f'{m.group(1)}{POOL_FRAC}', txt, count=1)  # first match = dev block
assert n == 1
open('configs/baseline.yaml', 'w').write(txt)
!grep -n "pool_sample_frac" configs/baseline.yaml

## Run A: baseline (current generator + prune)

In [ ]:
run('A_baseline', 'colab A: baseline generator + prune')

## Run B: capped exact_name variant (friend's generator + prune)

In [ ]:
shutil.copy('src/blocking/candidate_generator.py', 'results/candidate_generator_baseline.py')
shutil.copy('variants/candidate_generator_capped.py', 'src/blocking/candidate_generator.py')
run('B_capped', 'colab B: capped exact_name (exact_name_cap=8, boost=50) + prune')
# restore baseline generator so the working copy is left as it was
shutil.copy('results/candidate_generator_baseline.py', 'src/blocking/candidate_generator.py')

In [ ]:
import pandas as pd
def flat(d, p=''):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict): out.update(flat(v, f'{p}{k}.'))
        elif isinstance(v, (int, float, str)): out[f'{p}{k}'] = v
    return out
A = flat(json.load(open('results/A_baseline_metrics.json')))
B = flat(json.load(open('results/B_capped_metrics.json')))
df = pd.DataFrame({'A_baseline': A, 'B_capped': B})
df['B-A'] = pd.to_numeric(df.B_capped, errors='coerce') - pd.to_numeric(df.A_baseline, errors='coerce')
pd.set_option('display.max_rows', 200, 'display.width', 160)
df

In [ ]:
# save everything back to Drive
shutil.make_archive('/content/drive/MyDrive/MLChallenge/ab_results', 'zip', 'results')
print('saved MyDrive/MLChallenge/ab_results.zip')